# apmc_page_split.ipynb -- catalog PDF splitter (deterministic rule + vision fallback)

Splits an ASCC catalog PDF into marking-anchored half-page chunks.

## Pass-1 design (v3)

ASCC catalog pages have a single thin printed VERTICAL RULE separating the two
text columns. The rule is full-height and high-contrast, so we detect it
deterministically with NumPy column-darkness profiling -- no vision call needed
for the common case. A Sonnet vision call confirms single-column pages only when
the deterministic detector finds no rule. Page numbers are still read by Sonnet
from a stitched header+footer strip.

This replaces the v2 design where Sonnet picked left/right gutter edges from a
center-cropped strip. v2 had a systematic left-bias on pages where the left
column contained large handstamp illustrations near the gutter -- the model
locked onto "where the left column glyphs end" rather than the printed rule.

## Inputs / outputs

- input PDF:        `wip/in/{BASENAME}.pdf`
- rendered pages:   `wip/cache/{BASENAME}_full/page-NNNN.png`     (NNNN = PDF index)
- pass-1 halves:    `wip/cache/{BASENAME}_halves/page-NNNN-{L,R}.png` (NNNN = catalog page)
- pass-1 cache:     `wip/cache/{BASENAME}_split_halves.json`
- pass-2 cache:     `wip/cache/{BASENAME}_split_chunks.json`
- final chunks:     `wip/out/{BASENAME}/page-NNNN-{L,R}-K.png`     (or `page-NNNN-K.png` for single-col)
- manifest CSV:     `wip/out/{BASENAME}_split_manifest.csv`

## Pipeline

1. Render PDF with pdftoppm at 300 DPI to `FULL_DIR` (cached).
2. Pass 1a: deterministic rule detection per page (free, no API call).
3. Pass 1b: Sonnet page-number call per page; Sonnet single-column-confirm call
   only when the deterministic detector returned no rule.
4. Crop each full page into L/R halves at the detected rule x-coordinate; write
   to `HALVES_DIR` named by catalog page.
5. Pass 2: one vision call per half image to detect y-cuts anchored to postal
   markings.
6. Crop each half into chunks using filtered cuts; write to `OUTPUT_DIR`.
7. Emit manifest CSV.

## Downstream

`apmc_page_extract.ipynb` currently expects `page-NNNN-{L,R}.png`. New filenames
from pass 2 are `page-NNNN-{L,R}-K.png` and `page-NNNN-K.png`. Update its
NAME_RE and per-page iteration before running it against this output.

In [ ]:
import base64
import json
import os
import re
import shutil
import subprocess
import sys
from io import BytesIO
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from dotenv import load_dotenv
import anthropic

load_dotenv(Path('..') / '.env')

BASENAME             = 'VA_ASCC_CTLG'
PDF_PATH             = Path(f'./wip/in/{BASENAME}.pdf')
FULL_DIR             = Path(f'./wip/cache/{BASENAME}_full')
HALVES_DIR           = Path(f'./wip/cache/{BASENAME}_halves')
HALVES_CACHE         = Path(f'./wip/cache/{BASENAME}_split_halves.json')
CHUNKS_CACHE         = Path(f'./wip/cache/{BASENAME}_split_chunks.json')
OUTPUT_DIR           = Path(f'./wip/out/{BASENAME}')
MANIFEST_CSV         = Path(f'./wip/out/{BASENAME}_split_manifest.csv')

DPI                  = 300
MIN_CHUNK_HEIGHT_PX  = 200

# Pass-1 deterministic rule detector parameters.
# The printed rule sits very near width//2. Search a generous band, but assert
# the final result lands within RULE_CENTER_TOLERANCE of true centre so we
# notice if the heuristic ever drifts.
RULE_SEARCH_FRACTION   = 0.20   # search +/- 10% of width around centre
RULE_BAND_TOP_FRAC     = 0.20   # ignore top 20% of page (header)
RULE_BAND_BOTTOM_FRAC  = 0.85   # ignore bottom 15% of page (footer)
RULE_DARK_THRESHOLD    = 128    # pixel value below which we count as 'dark'
RULE_MIN_DARK_FRACTION = 0.70   # column must be dark for >=70% of band height
RULE_CENTER_TOLERANCE  = 0.05   # rule must lie within +/- 5% of width//2

# Pass-1 vision strip sizes (300 DPI -> 1 inch = 300 px).
HEADER_STRIP_HEIGHT  = 250   # ~0.83" tall band from top
FOOTER_STRIP_HEIGHT  = 250   # ~0.83" tall band from bottom
HEADER_FOOTER_SEP_PX = 6     # black separator between header and footer in composite

# Pass 1 vision: page-number every page, single-column-confirm only when
# deterministic detector found no rule. Cache invalidates on prompt-version bump.
HALVES_MODEL         = 'claude-sonnet-4-6'
HALVES_PROMPT_VER    = 'v3'  # v3 = deterministic rule + vision fallback
HALVES_MAX_TOKENS    = 128

CHUNKS_MODEL         = 'claude-opus-4-7'
CHUNKS_PROMPT_VER    = 'v2'  # v2 = entry-aware pass-2 prompt
CHUNKS_MAX_TOKENS    = 1024  # generous: room for many cuts + any preamble

FORCE_REFRESH_RENDER = False
FORCE_REFRESH_HALVES = False
FORCE_REFRESH_CHUNKS = False

assert PDF_PATH.is_file(), f'missing {PDF_PATH}'
assert shutil.which('pdftoppm'), 'pdftoppm not on PATH (install poppler)'
assert os.environ.get('ANTHROPIC_API_KEY'), 'ANTHROPIC_API_KEY not set'

for d in (FULL_DIR, HALVES_DIR, OUTPUT_DIR,
          HALVES_CACHE.parent, CHUNKS_CACHE.parent):
    d.mkdir(parents=True, exist_ok=True)

client = anthropic.Anthropic()
print(f'PDF:           {PDF_PATH}')
print(f'FULL_DIR:      {FULL_DIR}')
print(f'HALVES:        {HALVES_DIR}')
print(f'OUT:           {OUTPUT_DIR}')
print(f'HALVES_MODEL:  {HALVES_MODEL}  ({HALVES_PROMPT_VER})')
print(f'CHUNKS_MODEL:  {CHUNKS_MODEL}  ({CHUNKS_PROMPT_VER})')


In [ ]:
# Render PDF -> wip/cache/{BASENAME}_full/page-NNNN.png
# NNNN is the PDF order index, zero-padded to 4 digits.
# Skip if FULL_DIR already has page-*.png and not FORCE_REFRESH_RENDER.

def _idx(p):
    m = re.search(r'-(\d+)\.png$', p.name)
    return int(m.group(1)) if m else 0

def render_pdf(pdf_path, full_dir, dpi):
    # pdftoppm writes <prefix>-N.png un-padded for low N -- render to a sentinel
    # prefix, then rename to zero-padded page-NNNN.png in numeric order.
    for old in full_dir.glob('_render-*.png'):
        old.unlink()
    prefix = full_dir / '_render'
    subprocess.run(
        ['pdftoppm', '-r', str(dpi), '-png', str(pdf_path), str(prefix)],
        check=True,
    )
    rendered = sorted(full_dir.glob('_render-*.png'), key=_idx)
    pages = []
    for i, src in enumerate(rendered, 1):
        dst = full_dir / f'page-{i:04d}.png'
        if dst.exists():
            dst.unlink()
        src.rename(dst)
        pages.append(dst)
    return pages

existing = sorted(FULL_DIR.glob('page-*.png'), key=_idx)
if existing and not FORCE_REFRESH_RENDER:
    print(f'render: {len(existing)} pages already in {FULL_DIR}, skipping pdftoppm')
    full_pages = existing
else:
    print(f'render: pdftoppm -r {DPI} {PDF_PATH.name} -> {FULL_DIR}')
    full_pages = render_pdf(PDF_PATH, FULL_DIR, DPI)
    print(f'render: wrote {len(full_pages)} pages')

assert full_pages, 'no rendered pages -- check PDF and pdftoppm output'


## Pass 1 -- rule detection + page number

Two sub-tasks per page, ordered cheapest-first:

1. **Rule detection (deterministic, free).** Profile column darkness in the
   middle vertical band of the page and locate the single full-height column
   that registers as dark for the bulk of the band. ASCC catalog pages have a
   thin printed vertical rule separating the columns; it is by far the
   strongest dark column near the page midpoint. Returns `rule_x` or `None`.

2. **Page number (vision call).** Send a composite of the top
   `HEADER_STRIP_HEIGHT` and bottom `FOOTER_STRIP_HEIGHT` pixels of the page
   (stacked, separated by a thin black bar). Ask only for `{page_number: int}`.

3. **Single-column confirm (vision call, fallback only).** Triggers only when
   step 1 found no rule. Send the full page rendered down and ask
   `{has_two_columns: bool}`. This is rare -- plate pages, blanks, dividers.

Frozen system prompts sent with `cache_control: ephemeral`.

In [ ]:
PAGE_NUMBER_SYSTEM_PROMPT = """You receive a stitched image showing the TOP margin and BOTTOM margin of one page of an old American Stampless Cover (ASCC) catalog. The two margins are stacked vertically with a black separator bar between them.

Your job is to read the printed catalog page number. It appears as a plain integer somewhere in either margin -- usually in the footer, occasionally in the header.

Return STRICT JSON only -- no markdown, no prose, no code fences:

  {"page_number": 419}

Rules:
- page_number is an integer >= 1.
- Never return null. If you cannot read it confidently, return your best integer guess.
- Output JSON only."""

SINGLE_COL_SYSTEM_PROMPT = """You receive one page from an old American Stampless Cover (ASCC) catalog.

A typical page has TWO text columns separated by a thin printed vertical rule running most of the page height. Some pages do NOT have that two-column layout -- they may be a full-page plate of postmark illustrations, a section divider, a blank page, or a single-column body of text with no rule.

Your job is to decide whether this page is laid out as TWO columns separated by a printed vertical rule.

Return STRICT JSON only -- no markdown, no prose, no code fences:

  {"has_two_columns": true}

or

  {"has_two_columns": false}

Output JSON only."""

print(f'page-number prompt:    {len(PAGE_NUMBER_SYSTEM_PROMPT)} chars')
print(f'single-column prompt:  {len(SINGLE_COL_SYSTEM_PROMPT)} chars')


In [ ]:
def _img_to_b64_png(im):
    buf = BytesIO()
    im.save(buf, format='PNG')
    return base64.standard_b64encode(buf.getvalue()).decode()

def _parse_strict_json(resp):
    text = resp.content[0].text.strip()
    if text.startswith('```'):
        text = re.sub(r'^```(?:json)?\s*|\s*```$', '', text, flags=re.DOTALL)
    return json.loads(text)


# ---- 1a: deterministic rule detection -------------------------------------

def detect_rule_x(im):
    """Locate the printed vertical rule separating the two text columns.

    Profiles column darkness across a band that excludes header and footer
    (where the running header underline and page-number area introduce
    horizontal dark structure that would confuse the column-wise count).
    Returns x in full-page coords, or None if no column qualifies.
    """
    arr = np.asarray(im.convert('L'))
    h, w = arr.shape

    band_top    = int(h * RULE_BAND_TOP_FRAC)
    band_bottom = int(h * RULE_BAND_BOTTOM_FRAC)
    band = arr[band_top:band_bottom, :]
    band_h = band.shape[0]

    cx = w // 2
    half_search = int(w * RULE_SEARCH_FRACTION / 2)
    x_lo = max(0, cx - half_search)
    x_hi = min(w, cx + half_search)

    col_dark = (band[:, x_lo:x_hi] < RULE_DARK_THRESHOLD).sum(axis=0)
    min_dark = int(band_h * RULE_MIN_DARK_FRACTION)
    qualifying = np.where(col_dark >= min_dark)[0]

    if qualifying.size == 0:
        return None

    # Multiple adjacent columns will qualify because the rule is several
    # pixels thick at 300 DPI. Take the median to pick the rule centre.
    rule_x = x_lo + int(np.median(qualifying))

    # Sanity: must lie close to the page midpoint.
    if abs(rule_x - cx) > w * RULE_CENTER_TOLERANCE:
        return None

    return rule_x


# ---- 1b: vision page-number ------------------------------------------------

def build_header_footer_strip(im):
    w, h = im.size
    head = im.crop((0, 0, w, min(h, HEADER_STRIP_HEIGHT)))
    foot = im.crop((0, max(0, h - FOOTER_STRIP_HEIGHT), w, h))
    out_h = head.size[1] + HEADER_FOOTER_SEP_PX + foot.size[1]
    out = Image.new('RGB', (w, out_h), (0, 0, 0))
    out.paste(head, (0, 0))
    out.paste(foot, (0, head.size[1] + HEADER_FOOTER_SEP_PX))
    return out

def detect_page_number(header_footer_im):
    img_b64 = _img_to_b64_png(header_footer_im)
    user = [
        {'type': 'image',
         'source': {'type': 'base64', 'media_type': 'image/png', 'data': img_b64}},
        {'type': 'text', 'text': 'Read the printed catalog page number. Return JSON only.'},
    ]
    resp = client.messages.create(
        model=HALVES_MODEL,
        max_tokens=HALVES_MAX_TOKENS,
        system=[{'type': 'text', 'text': PAGE_NUMBER_SYSTEM_PROMPT,
                 'cache_control': {'type': 'ephemeral'}}],
        messages=[{'role': 'user', 'content': user}],
    )
    data = _parse_strict_json(resp)
    pn = data['page_number']
    assert isinstance(pn, int) and pn >= 1, f'bad page_number {pn!r}'
    return pn


# ---- 1c: vision single-column fallback (only when rule_x is None) ---------

def confirm_single_column(im):
    img_b64 = _img_to_b64_png(im)
    user = [
        {'type': 'image',
         'source': {'type': 'base64', 'media_type': 'image/png', 'data': img_b64}},
        {'type': 'text', 'text': 'Is this page laid out as two columns with a printed vertical rule? Return JSON only.'},
    ]
    resp = client.messages.create(
        model=HALVES_MODEL,
        max_tokens=HALVES_MAX_TOKENS,
        system=[{'type': 'text', 'text': SINGLE_COL_SYSTEM_PROMPT,
                 'cache_control': {'type': 'ephemeral'}}],
        messages=[{'role': 'user', 'content': user}],
    )
    data = _parse_strict_json(resp)
    htc = data['has_two_columns']
    assert isinstance(htc, bool), f'bad has_two_columns {htc!r}'
    return htc


In [ ]:
# Pass 1 cache schema (v3):
#   {
#     "model": "claude-sonnet-4-6",
#     "prompt_version": "v3",
#     "responses": {
#       "pdf-page-0001": {"page_number": 419, "has_two_columns": true,
#                         "rule_x": 1009, "rule_source": "deterministic",
#                         "image_width": 2024, "image_height": 2970}
#     }
#   }
#
# rule_source: "deterministic" | "vision_single_col" | "vision_single_col_failed"
#   - deterministic: detect_rule_x returned an x; has_two_columns=true.
#   - vision_single_col: detect_rule_x returned None; vision confirmed single-col.
#   - vision_single_col_failed: detect_rule_x returned None but vision says
#     two-column; rule_x is set to width//2 as a fallback. RAISES so that
#     a human can investigate -- this should be rare and worth a look.

if HALVES_CACHE.exists() and not FORCE_REFRESH_HALVES:
    halves_cache = json.loads(HALVES_CACHE.read_text())
    if (halves_cache.get('model') != HALVES_MODEL
            or halves_cache.get('prompt_version') != HALVES_PROMPT_VER):
        print('halves cache: model/prompt_version mismatch, invalidating')
        halves_cache = {'model': HALVES_MODEL, 'prompt_version': HALVES_PROMPT_VER, 'responses': {}}
else:
    halves_cache = {'model': HALVES_MODEL, 'prompt_version': HALVES_PROMPT_VER, 'responses': {}}

calls = 0
rule_failures = []
for pdf_idx, full_png in enumerate(full_pages, 1):
    key = f'pdf-page-{pdf_idx:04d}'
    with Image.open(full_png) as im:
        iw, ih = im.size
        if key in halves_cache['responses'] and not FORCE_REFRESH_HALVES:
            rec = halves_cache['responses'][key]
        else:
            # 1a: deterministic rule detection (free)
            rule_x = detect_rule_x(im)

            # 1b: page number (vision)
            hf_strip = build_header_footer_strip(im)
            pn = detect_page_number(hf_strip)
            calls += 1

            # 1c: single-column confirmation (vision, fallback only)
            if rule_x is not None:
                htc = True
                rule_source = 'deterministic'
            else:
                htc = confirm_single_column(im)
                calls += 1
                if htc:
                    # Two columns claimed but rule detector missed -- record
                    # for end-of-loop reporting; assign centre as best guess.
                    rule_x = iw // 2
                    rule_source = 'vision_single_col_failed'
                    rule_failures.append((key, pn))
                else:
                    rule_x = -1
                    rule_source = 'vision_single_col'

            rec = {
                'page_number':     pn,
                'has_two_columns': htc,
                'rule_x':          rule_x,
                'rule_source':     rule_source,
                'image_width':     iw,
                'image_height':    ih,
            }
            halves_cache['responses'][key] = rec
            HALVES_CACHE.write_text(json.dumps(halves_cache, indent=2))

    pn = rec['page_number']
    if rec['has_two_columns']:
        print(f'  {key} -> catalog {pn:>4d}  rule_x={rec["rule_x"]:>5d}  '
              f'source={rec["rule_source"]:<28s}  size {iw}x{ih}')
    else:
        print(f'  {key} -> catalog {pn:>4d}  SINGLE-COLUMN  ({rec["rule_source"]})  size {iw}x{ih}')

print(f'pass-1 vision calls made: {calls}')

if rule_failures:
    print()
    print('=== WARNING: deterministic rule detector failed on these two-column pages ===')
    for key, pn in rule_failures:
        print(f'  {key} catalog {pn}')
    print('rule_x has been set to image_width//2 as a fallback; inspect the')
    print('halves output and hand-edit the cache JSON if the split is off.')


In [ ]:
# Crop full pages into L/R halves (or full single-col pages) at the rule x.
# Wipes HALVES_DIR/page-*.png first so stale halves do not linger across re-runs.
# Detects duplicate catalog page numbers BEFORE any writes -- raises with offenders listed.
#
# The rule itself is one column wide and belongs to neither column visually,
# so we crop EXACTLY at rule_x with no overlap and no pad. Compare to v2,
# which produced a left-half ending at left_edge+12 and a right-half starting
# at right_edge-12 -- needed only because v2 had a two-edge gutter band.

by_pn = {}
for pdf_idx, full_png in enumerate(full_pages, 1):
    key = f'pdf-page-{pdf_idx:04d}'
    pn = halves_cache['responses'][key]['page_number']
    by_pn.setdefault(pn, []).append(key)
dups = {pn: keys for pn, keys in by_pn.items() if len(keys) > 1}
if dups:
    raise RuntimeError(
        f'duplicate catalog page_number(s) detected in pass-1 cache: {dups}.\n'
        f'Edit {HALVES_CACHE} to fix the misread page_number(s) and re-run this cell.'
    )

for old in HALVES_DIR.glob('page-*.png'):
    old.unlink()

halves_written = 0
for pdf_idx, full_png in enumerate(full_pages, 1):
    key = f'pdf-page-{pdf_idx:04d}'
    rec = halves_cache['responses'][key]
    pn = rec['page_number']
    with Image.open(full_png) as im:
        w, h = im.size
        if rec['has_two_columns']:
            rx = rec['rule_x']
            assert 0 < rx < w, f'rule_x {rx} out of (0, {w}) for {key}'
            im.crop((0,  0, rx, h)).save(HALVES_DIR / f'page-{pn:04d}-L.png')
            im.crop((rx, 0, w,  h)).save(HALVES_DIR / f'page-{pn:04d}-R.png')
            halves_written += 2
        else:
            im.crop((0, 0, w, h)).save(HALVES_DIR / f'page-{pn:04d}.png')
            halves_written += 1

print(f'wrote {halves_written} half images to {HALVES_DIR}')
print('--- INSPECT THESE BEFORE PROCEEDING TO PASS 2 ---')
print('Open a few halves and confirm no listing text or marking is clipped at the seam.')


## Pass 2 -- chunk cuts per half (v2 prompt)

One Sonnet vision call per half image. Returns `{cuts: [int, ...]}` in that
half's local y-coord system.

The v2 prompt teaches the catalog's entry structure explicitly: each entry is a
postal MARKING followed by its LISTING beneath it (marking-on-top, listing-
below). Cuts go between the END of one entry's listing and the START of the
next entry's marking -- i.e. between listing-N and marking-(N+1). The v1 prompt
left this implicit and the model defaulted to the more common label-then-
illustration reading order, producing chunks with markings at the BOTTOM.

In [ ]:
CHUNKS_SYSTEM_PROMPT = """You are a layout analyzer for ONE column-image from an old American Stampless Cover (ASCC) catalog.

This is a STAMPLESS COVER catalog. The images on these pages are postal markings -- postmarks, handstamps, and manuscript markings. They are NEVER adhesive postage stamps.

You will receive ONE half-column image (or a full single-column page). Your job is to find clean horizontal cut lines so that downstream extraction can process the column as smaller chunks, with each catalog ENTRY in its own chunk.

== Entry structure (READ CAREFULLY) ==

Each catalog entry has this order, top-to-bottom:

  1. An ILLUSTRATION OF THE POSTAL MARKING -- a postmark, handstamp, or manuscript marking.
  2. The LISTING for that marking -- one or more lines of text describing dates, sizes, colors, values. Listings end with a price like "1500.00" preceded by leader dots, or with "---" for unpriced entries.

The marking comes FIRST. Its listing comes BELOW it.

Multiple entries stack vertically:

  marking A
  listing A
  marking B
  listing B
  marking C
  listing C
  ...

Markings vary in height. Small manuscripts may be 50-80 px tall; elaborate manuscripts with descenders or ornaments may exceed 150 px. Do not assume a tall ink block is two separate markings -- it may be one marking with a long vertical reach.

== Cut placement rule ==

A cut goes in the WHITE SPACE between:
  - the LAST LINE of one entry's listing, and
  - the START of the next entry's marking.

Concretely: cut between listing-N (end) and marking-(N+1) (start).

Do NOT cut between a marking and its own listing. Marking-N and listing-N belong in the SAME chunk.

== Edge cases ==

- TOP-FLUSH MARKING: if the FIRST marking on the image sits flush at the top with no preceding listing on this image, emit no cut for that marking. It already sits at the top of chunk 1 naturally. (The chunk may also contain a state heading or section subheading above the first marking -- that is fine; those headings ride along.)

- BOTTOM-FLUSH ENTRY: if the LAST entry's listing runs to the bottom of the image with nothing else after it, no special handling -- the previous cut already isolated it.

- LISTING-ONLY ENTRIES: occasionally a listing line appears with NO marking above it (a sub-entry that shares the previous entry's marking, or a historical note like "The British evacuated Norfolk in December 1775"). It rides with the PRECEDING marking's chunk. Do not cut between the previous listing and a listing-only entry.

- INLINE ANNOTATIONS: small date or rate annotations printed directly next to a marking are part of the SAME marking image, not separate ones.

- SECTION HEADINGS: typeset banners like "AMERICAN CONGRESS AND CONFEDERATION POSTMARKS" are not markings and not listings. They ride with whichever chunk they happen to land in.

== Output format ==

Return STRICT JSON only -- no markdown, no prose, no code fences -- of this exact shape:

  {"cuts": [820, 1640]}

Or, for an image with no valid cuts (e.g. only one entry, or all markings top-flush):

  {"cuts": []}

Coordinate system: y-pixels are integers in the LOCAL coordinate system of THIS image, with origin top-left. All cuts must be strictly inside (0, image_height) and strictly ascending.

Output JSON only."""

print(f'pass-2 prompt: {len(CHUNKS_SYSTEM_PROMPT)} chars')


In [ ]:
_LAST_RESPONSES = {}  # stem -> raw text, for post-hoc inspection

def detect_cuts(image_path, image_height):
    img_b64 = base64.standard_b64encode(image_path.read_bytes()).decode()
    user = [
        {'type': 'image',
         'source': {'type': 'base64', 'media_type': 'image/png', 'data': img_b64}},
        {'type': 'text',
         'text': f'Image height: {image_height} px. Output JSON only -- no preamble, no markdown, no code fences.'},
    ]
    resp = client.messages.create(
        model=CHUNKS_MODEL,
        max_tokens=CHUNKS_MAX_TOKENS,
        system=[{'type': 'text', 'text': CHUNKS_SYSTEM_PROMPT,
                 'cache_control': {'type': 'ephemeral'}}],
        messages=[{'role': 'user', 'content': user}],
    )

    text = resp.content[0].text.strip() if resp.content else ''
    _LAST_RESPONSES[image_path.stem] = {
        'stop_reason': resp.stop_reason,
        'usage_in':   resp.usage.input_tokens,
        'usage_out':  resp.usage.output_tokens,
        'text':       text,
    }

    if resp.stop_reason == 'max_tokens':
        raise RuntimeError(
            f'{image_path.stem}: response truncated at {resp.usage.output_tokens} tokens '
            f'(max_tokens={CHUNKS_MAX_TOKENS}). Bump CHUNKS_MAX_TOKENS in cell a1 and re-run. '
            f'Raw text: {text!r}'
        )

    if text.startswith('```'):
        text = re.sub(r'^```(?:json)?\s*|\s*```$', '', text, flags=re.DOTALL)

    # Sometimes the model produces preamble before the JSON. Pull out the first
    # {...} object as a fallback.
    if not text.startswith('{'):
        m = re.search(r'\{.*\}', text, flags=re.DOTALL)
        if m:
            text = m.group(0)

    try:
        data = json.loads(text)
    except json.JSONDecodeError as e:
        raise RuntimeError(
            f'{image_path.stem}: could not parse JSON from response. '
            f'stop_reason={resp.stop_reason}, output_tokens={resp.usage.output_tokens}. '
            f'Raw text: {text!r}'
        ) from e

    cuts = data['cuts']
    assert isinstance(cuts, list), f'cuts not a list: {cuts!r}'
    for i, c in enumerate(cuts):
        assert isinstance(c, int), f'cuts[{i}] not int: {c!r}'
        assert 0 < c < image_height, f'cuts[{i}]={c} out of (0, {image_height})'
    assert cuts == sorted(cuts), f'cuts not strictly ascending: {cuts}'
    for a, b in zip(cuts, cuts[1:]):
        assert a != b, f'cuts has duplicate: {cuts}'
    return cuts


In [ ]:
# Pass 2 cache schema:
#   {
#     "model": "claude-opus-4-7",
#     "prompt_version": "v1",
#     "responses": {
#       "page-0419-L": {"cuts": [820, 1640], "image_height": 3300},
#       "page-0419-R": {"cuts": [350],       "image_height": 3300},
#       "page-0250":   {"cuts": [],          "image_height": 3300}
#     }
#   }

if CHUNKS_CACHE.exists() and not FORCE_REFRESH_CHUNKS:
    chunks_cache = json.loads(CHUNKS_CACHE.read_text())
    if (chunks_cache.get('model') != CHUNKS_MODEL
            or chunks_cache.get('prompt_version') != CHUNKS_PROMPT_VER):
        print('chunks cache: model/prompt_version mismatch, invalidating')
        chunks_cache = {'model': CHUNKS_MODEL, 'prompt_version': CHUNKS_PROMPT_VER, 'responses': {}}
else:
    chunks_cache = {'model': CHUNKS_MODEL, 'prompt_version': CHUNKS_PROMPT_VER, 'responses': {}}

halves = sorted(HALVES_DIR.glob('page-*.png'))
assert halves, f'no halves under {HALVES_DIR} -- run pass-1 cells first'

calls = 0
for half_png in halves:
    stem = half_png.stem
    with Image.open(half_png) as im:
        ih = im.size[1]
    if stem in chunks_cache['responses'] and not FORCE_REFRESH_CHUNKS:
        rec = chunks_cache['responses'][stem]
    else:
        cuts = detect_cuts(half_png, ih)
        rec = {'cuts': cuts, 'image_height': ih}
        chunks_cache['responses'][stem] = rec
        CHUNKS_CACHE.write_text(json.dumps(chunks_cache, indent=2))
        calls += 1
    print(f'  {stem}  cuts={rec["cuts"]}')

print(f'pass-2 calls made: {calls}')


In [ ]:
# Crop each half into chunks using filtered cuts.
# Wipes OUTPUT_DIR/*.png first so stale chunks do not linger across re-runs.

def filter_cuts(cuts, image_height, label):
    kept = []
    last = 0
    for c in cuts:
        gap_above = c - last
        gap_below = image_height - c
        if gap_above < MIN_CHUNK_HEIGHT_PX:
            print(f'    [{label}] drop cut y={c}: gap above={gap_above} < {MIN_CHUNK_HEIGHT_PX}')
            continue
        if gap_below < MIN_CHUNK_HEIGHT_PX:
            print(f'    [{label}] drop cut y={c}: gap below={gap_below} < {MIN_CHUNK_HEIGHT_PX}')
            continue
        kept.append(c)
        last = c
    return kept

def write_chunks(half_png, raw_cuts, out_dir):
    stem = half_png.stem
    with Image.open(half_png) as im:
        w, h = im.size
        kept = filter_cuts(raw_cuts, h, stem)
        ys = [0] + kept + [h]
        for k, (y0, y1) in enumerate(zip(ys[:-1], ys[1:]), 1):
            im.crop((0, y0, w, y1)).save(out_dir / f'{stem}-{k}.png')
    return kept, len(ys) - 1

for old in OUTPUT_DIR.glob('*.png'):
    old.unlink()

summary = []
total_chunks = 0
for half_png in halves:
    stem = half_png.stem
    rec = chunks_cache['responses'][stem]
    kept, n = write_chunks(half_png, rec['cuts'], OUTPUT_DIR)
    total_chunks += n
    summary.append({'stem': stem, 'raw_cuts': rec['cuts'], 'kept_cuts': kept,
                    'num_chunks': n, 'image_height': rec['image_height']})

print(f'wrote {total_chunks} chunk PNGs across {len(halves)} halves -> {OUTPUT_DIR}')


In [ ]:
# Manifest CSV: one row per half image (or per page for single-col).
#   pdf_page_index, catalog_page, side, has_two_columns, rule_x, rule_source,
#   half_width, half_height, raw_cuts, kept_cuts, num_chunks

STEM_RE = re.compile(r'^page-(\d{4})(?:-([LR]))?$')

pn_to_pdf_idx = {}
for pdf_idx, full_png in enumerate(full_pages, 1):
    key = f'pdf-page-{pdf_idx:04d}'
    pn_to_pdf_idx[halves_cache['responses'][key]['page_number']] = pdf_idx

rows = []
for s in summary:
    m = STEM_RE.match(s['stem'])
    assert m, f'unexpected stem {s["stem"]}'
    pn = int(m.group(1))
    side = m.group(2) or ''
    pdf_idx = pn_to_pdf_idx[pn]
    h_rec = halves_cache['responses'][f'pdf-page-{pdf_idx:04d}']
    with Image.open(HALVES_DIR / f'{s["stem"]}.png') as im:
        hw, hh = im.size
    rows.append({
        'pdf_page_index':  pdf_idx,
        'catalog_page':    pn,
        'side':            side,
        'has_two_columns': h_rec['has_two_columns'],
        'rule_x':          h_rec['rule_x'],
        'rule_source':     h_rec['rule_source'],
        'half_width':      hw,
        'half_height':     hh,
        'raw_cuts':        ';'.join(str(c) for c in s['raw_cuts']),
        'kept_cuts':       ';'.join(str(c) for c in s['kept_cuts']),
        'num_chunks':      s['num_chunks'],
    })

df = pd.DataFrame(rows).sort_values(['pdf_page_index', 'side']).reset_index(drop=True)
df.to_csv(MANIFEST_CSV, index=False)
print(f'manifest: {MANIFEST_CSV}  ({len(df)} rows)')
df.head(10)


## Verification

- **Pass-1 sanity** (after cell a5): `pass-1 vision calls made` should equal
  `len(full_pages)` plus a small number for any single-column / fallback pages.
  Inspect the per-page log: `rule_source=deterministic` is the happy path. Any
  `rule_source=vision_single_col_failed` rows are flagged at the end of the
  loop -- open the corresponding halves and confirm; if the split is wrong,
  hand-edit `rule_x` in `_split_halves.json` and re-run from cell a6.

- **Halves spot-check** (after cell a6): open a few entries in
  `wip/cache/{BASENAME}_halves/` and confirm no listing text or postal marking
  is clipped at the L/R seam BEFORE letting pass 2 spend tokens. The rule
  itself is a thin printed line dividing the columns; either half can keep it
  or neither -- what matters is that nothing else crosses.

- **Pass-2 sanity** (after cell a9): `pass-2 calls made: 0` on re-runs.
  Spot-check chunks under `wip/out/{BASENAME}/`:
  - For each `page-NNNN-{L,R}-K` with K >= 2, the corresponding marking should
    sit at the TOP of the image, with listings flowing below it.
  - Chunk K=1 should END with listing text, not contain a marking already.
  - No chunk should be mostly whitespace (empty-chunk guard working).
  - Per-half chunk counts should be sane (typically 1-5, not 10).

- **Manifest sanity**: `catalog_page` strictly monotonic across
  `pdf_page_index`. Any non-monotonic row = pass-1 page-number misread; edit
  the entry in `_split_halves.json` by hand and re-run from cell a6 onward
  (cropping reads from cache, no API calls).

- **Re-tuning without API spend**: change `MIN_CHUNK_HEIGHT_PX`,
  `RULE_SEARCH_FRACTION`, `RULE_MIN_DARK_FRACTION`, etc., then re-run cell a5
  with `FORCE_REFRESH_HALVES = True` if you want to redo deterministic
  detection (still free), or just cell a6 if you only changed the crop. Pass 2
  vision calls only re-run when you bump `CHUNKS_PROMPT_VER` or set
  `FORCE_REFRESH_CHUNKS = True`.

- **Downstream refactor (out of scope)**: `apmc_page_extract.ipynb` currently
  uses `NAME_RE = r"^page-(\d{4})-([LR])\.png$"`. New filenames are
  `page-(\d{4})-([LR])-(\d+).png` (two-column) and `page-(\d{4})-(\d+).png`
  (single-column). Update the regex and per-page iteration to walk
  `(catalog_page, side, chunk_k)` order before running it against this output.